# Segmentation workflow for multichannel 3D images

OK. So we have seen how to segment and count objects using scikit-image and cellpose. Now we approach a new challenge; segmenting 3D objects. 

The general pipeline looks similar to what we saw earlier: ????????????????
1. Define object and constraints
2. Preprocess (illumination, noise, contrast)
    - with optional downsampling to reduce compute requirements when developing code.
3. Extract foreground
4. Generate seeds (if instances touch)
5. Separate objects (watershed)
6. Filter and correct
7. Quality-check and iterate

We segment using StarDist or Cellpose and have optional post-processing to remove objects, perform smoothing etc. Then we will counts the number of nuclei/cells per 3D image.

For quality control, we produce some visual outputs e.g. ????

In the next notebook, we will identify cell size and count subcellular structures in a multichannel analysis.

1. Define object and constraints
What to record up front
Object: we have in different channels nuclei, whole cells and sub-objects. Here, we segment and count nuclei.

Imaging: fluorescence

Size range: expected pixel area or diameter. Calculate:

pixel size = 512 / 246 (from metadata) = 2um
expected_pixel_diameter = real_diameter / pixel_diameter = 15 (from google search) / 2 = 7.5 pixels

Touching frequency: low (for nuclei)

Shape: round

Signal reliability: consistent

Error tolerance: more acceptable to miss objects or to over-split?

In [ ]:
## Libraries
## Load packages
# import glob, os
import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
import tifffile


### How many images to segment? More takes longer but gives more info
num_images_to_segment = 5

## set dirs
cwd = Path.cwd()

# Set the main project directory by looking upwards for repo name "UCL-Biosciences-Image-Analysis"
# Find the first parent that matches the target folder
for p in cwd.parents:
    if p.name == "UCL-Biosciences-Image-Analysis":
        DIR = p
        break
else:
    raise FileNotFoundError("Base directory 'UCL-Biosciences-Image-Analysis' not found in path.")

## Set key variables
input_folder = DIR / "input_data" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF"
output_folder = DIR / "output" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF"
Path(output_folder).mkdir(parents=True, exist_ok=True)

### Input data setup
We specify the **input folder** containing raw images above.

1. Define a dictionary (*channel_map*) to tell the pipeline which channel corresponds to which biological compartment:

    - `"nucleus": 0` → channel index 0 contains the nuclear stain.

    - `"cytoplasm": 2` → channel index 2 contains cytoplasmic signal.

    - `"intracellular": 1` → channel index 1 contains the organelle/structure of interest.

    This mapping depends on the acquisition setup and might change between datasets — check the raw image metadata if in doubt (e.g. via opening a representative example image in ImageJ)

2. Make sure voxel size (`voxel_size_um`) is set correctly for the dataset, since this is required for volume and size calculations
3. Finally, we use `load_multichannel_images()` to read the dataset into memory

    - Returns a list of volumes, each represented as a dictionary with:

        - "filename" → original file name.

        - "channels" → dictionary of channels as (Z, Y, X) arrays.

    - The preview (print) shows how to access data for a single image (e.g. `all_volumes[0]["channels"]["nucleus"]`).

In [ ]:
# 1. load images as multichannel dictionary

# add a dictionary for your images, stating which channel corresponds to which of the structures
channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

# enter voxel size of your images (obtained from the metadata)
voxel_size_um = [1, 0.48, 0.48 ] # Z, Y, X; from metadata

# loading images from the input folder directory
files = sorted(input_folder.glob("*.tif"))

all_volumes = []

for f in files:
    img = tifffile.imread(f)  # could be (Z,Y,X,C) or (C,Z,Y,X)

    # map channels to names
    channels_dict = {}
    for name, idx in channel_map.items():
        channels_dict[name] = zyx_channels[..., idx]

    # append structured entry with filename and channels (per image)
    all_volumes.append({
        "filename": f.stem, 
        "channels": channels_dict
    })

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # what channels are included in the dictionary? should be: dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

In [ ]:
f = files[0]
img = tifffile.imread(f)  # could be (Z,Y,X,C) or (C,Z,Y,X)


#### Quick check: middle slice for each channel  

Here we display the **middle Z-slice** of each channel from the same (first) image of the folder. You can view different images by editing `Volume_number`.

This allows you to quickly confirm:  
- Channel order (e.g., nucleus, cytoplasm, intracellular)
- The objects in the different channels should overlap, with nucleus and intracellular objects within the cytoplasm area.
- That the data is loaded correctly and matches expectations  

If the signal looks wrong (e.g., nucleus is empty but cytoplasm is bright), check the `channel_map` definition above.